In [1]:
import torch
from torch import nn
from kerops.ops.conv import Conv3d

In [2]:
import itertools
from tqdm.notebook import tqdm
from time import perf_counter, sleep
import numpy as np


def mean_std_percentile(x, lo=20, hi=80):
    x = np.asarray(x, dtype=np.float32)

    p_lo, p_hi = np.percentile(x, [lo, hi])

    mask = (x >= p_lo) & (x <= p_hi)
    x_mid = x[mask]

    return x_mid.mean(), x_mid.std()


def bench(func, *args, warmup=25, sleep_ms=100, n_iters=50, q_show=10, quantiles=(20, 80), **specset):
    results = []

    keys = list(specset.keys())
    values = list(specset.values())
    configs = list(itertools.product(*values))

    for config in tqdm(configs, desc="Benchmark configs"):
        if sleep_ms is not None:
            sleep(sleep_ms / 1000)
        kwargs = dict(zip(keys, config))

        try:
            func(*args, **kwargs)
            torch.cuda.synchronize()
        except Exception as e:
            pass

        for _ in range(warmup):
            func(*args, **kwargs)
        torch.cuda.synchronize()

        times_ms = []

        for _ in range(n_iters):
            start = perf_counter()
            func(*args, **kwargs)
            torch.cuda.synchronize()
            end = perf_counter()
            times_ms.append((end - start) * 1e3)

        mean, std = mean_std_percentile(times_ms, *quantiles)
        results.append({
            "spec": kwargs,
            "mean_ms": float(mean),
            "std_ms": float(std),
        })

    best_result = min(results, key=lambda x: x["mean_ms"])
    best_mean = best_result['mean_ms']
    best_std = best_result['std_ms']
    best_spec = best_result['spec']
    print(f'Best spec - {best_mean:.3f}+-{best_std:.3f}ms {best_spec}')

    good_results = []
    for result in results:
        if best_mean * (1 + q_show / 100) >= result["mean_ms"] and result['spec'] != best_spec:
            good_results.append(result)

    if good_results:
        print(30 * '-')
        print("Other good specs:")

        for result in good_results:
            print(f'{result['mean_ms']:.3f}+-{result['std_ms']:.3f}ms {result['spec']}')
    else:
        print(f'Other specs have a time difference of more than {q_show}%')

In [2]:
CIN = 64
COUT = 64
S = 96
D = 96

x = torch.randn(2, CIN, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
w = torch.randn(3, 3, 3, CIN, COUT, device='cuda', dtype=torch.float16)
conv = nn.Conv3d(CIN, COUT, 3, padding=1, bias=False, device='cuda')

In [13]:
%%timeit -r 10 -n 10
with torch.inference_mode(), torch.amp.autocast('cuda'):
    conv(x)

torch.cuda.synchronize()

5.96 ms ± 68 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [4]:
Conv3d(x, w)
torch.cuda.synchronize()

In [14]:
%%timeit -r 10 -n 10
Conv3d(x, w)
torch.cuda.synchronize()

6 ms ± 81.2 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [6]:
Conv3d(x, w, **{'num_warps': 2, 'D_BLOCK': 32, 'CIN_BLOCK': 32, 'LOAD_WEIGHT_FIRST': True, 'WEIGHT_MAJOR': True})
torch.cuda.synchronize()

In [7]:
%%timeit -r 10 -n 10
Conv3d(x, w, **{'num_warps': 2, 'D_BLOCK': 32, 'CIN_BLOCK': 32, 'LOAD_WEIGHT_FIRST': True, 'WEIGHT_MAJOR': True})
torch.cuda.synchronize()

5.86 ms ± 57.5 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [8]:
Conv3d(x, w, **{'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 16, 'LOAD_WEIGHT_FIRST': False, 'WEIGHT_MAJOR': True})
torch.cuda.synchronize()

In [12]:
%%timeit -r 10 -n 10
Conv3d(x, w, **{'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 16, 'LOAD_WEIGHT_FIRST': False, 'WEIGHT_MAJOR': True})
torch.cuda.synchronize()

5.93 ms ± 70.1 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [4]:
bench(Conv3d, x, w, num_warps=[2, 4], D_BLOCK=[16, 32], CIN_BLOCK=[16, 32, 64], impl=['default', 'wprior'], WEIGHT_FIRST=[True, False])

Benchmark configs:   0%|          | 0/48 [00:00<?, ?it/s]

Best spec - 5.835+-0.003ms {'num_warps': 2, 'D_BLOCK': 32, 'CIN_BLOCK': 32, 'impl': 'wprior', 'WEIGHT_FIRST': True}
------------------------------
Other good specs:
6.308+-0.017ms {'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 16, 'impl': 'default', 'WEIGHT_FIRST': True}
6.319+-0.217ms {'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 16, 'impl': 'wprior', 'WEIGHT_FIRST': True}
5.903+-0.045ms {'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 16, 'impl': 'wprior', 'WEIGHT_FIRST': False}
6.325+-0.093ms {'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 32, 'impl': 'wprior', 'WEIGHT_FIRST': True}
6.065+-0.118ms {'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 32, 'impl': 'wprior', 'WEIGHT_FIRST': False}
5.913+-0.009ms {'num_warps': 2, 'D_BLOCK': 16, 'CIN_BLOCK': 64, 'impl': 'wprior', 'WEIGHT_FIRST': True}
6.180+-0.001ms {'num_warps': 2, 'D_BLOCK': 32, 'CIN_BLOCK': 16, 'impl': 'default', 'WEIGHT_FIRST': True}
6.196+-0.001ms {'num_warps': 2, 'D_BLOCK': 32, 'CIN_BLOCK': 16, 'impl': 'default', 'WEIGHT_FIRST': Fals

In [63]:
grad_out = torch.randn_like(x)
weight = w.permute(-1, -2, 0, 1, 2).contiguous()

In [65]:
%%timeit -r 5 -n 5
_, grad_w, _ = torch.ops.aten.convolution_backward(
    grad_out,
    x,
    weight,
    [0],  # bias_sizes
    [1, 1, 1],  # stride
    [1, 1, 1],  # padding
    [1, 1, 1],  # dilation
    False,  # transposed
    [0, 0, 0],  # output padding
    1,  # groups!
    [True, True, False],  # output_mask - grad_inpt, grad_weight, grad_bias
)
torch.cuda.synchronize()

12.4 ms ± 140 μs per loop (mean ± std. dev. of 5 runs, 5 loops each)
